In [ ]:
# @title
%%writefile hello.cu
#include <stdio.h>

__global__ void hello(){
    printf("Hello from block: %u, thread: %u\n", blockIdx.x, threadIdx.x);
}

int main(){
    printf("Hello from CPU\n");
    hello<<<2, 2>>>();
    cudaError_t cudaStatus = cudaGetLastError();
   printf("CUDA launch result: %s\n", cudaGetErrorString(cudaStatus));
    cudaDeviceSynchronize();
}


Writing hello.cu


In [ ]:
# @title
# Компиляция с поддержкой CUDA 12.4
!nvcc -arch=compute_75 -code=sm_75 \
      -gencode arch=compute_75,code=sm_75 \
      hello.cu -o hello

# Запуск
!./hello


Hello from CPU
CUDA launch result: no error
Hello from block: 1, thread: 0
Hello from block: 1, thread: 1
Hello from block: 0, thread: 0
Hello from block: 0, thread: 1


In [ ]:
# @title
%%writefile vector_subtract.cu

#include <stdio.h>
#include <stdlib.h>
#include <time.h>

// CUDA kernel для поэлементного вычитания: C[i] = A[i] - B[i]
__global__ void vectorSubtract(const float *A, const float *B, float *C, int N) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    if (idx < N) {
        C[idx] = A[idx] - B[idx];
    }
}

// Функция для проверки результатов
bool verifyResult(const float *cpu_C, const float *gpu_C, int N) {
    for (int i = 0; i < N; i++) {
        if (cpu_C[i] != gpu_C[i]) {
            printf("Mismatch at index %d: CPU = %f, GPU = %f\n", i, cpu_C[i], gpu_C[i]);
            return false;
        }
    }
    return true;
}

int main() {
    int N;
    printf("Enter vector size (e.g., 1000000): ");
    scanf("%d", &N);

    // Выделение памяти на хосте
    size_t bytes = N * sizeof(float);
    float *h_A = (float*)malloc(bytes);
    float *h_B = (float*)malloc(bytes);
    float *h_C_cpu = (float*)malloc(bytes);
    float *h_C_gpu = (float*)malloc(bytes);

    // Инициализация случайными числами
    srand(time(NULL));
    for (int i = 0; i < N; i++) {
        h_A[i] = (float)(rand() % 100) / 10.0f;
        h_B[i] = (float)(rand() % 100) / 10.0f;
    }

    // CPU вычисления
    clock_t cpu_start = clock();
    for (int i = 0; i < N; i++) {
        h_C_cpu[i] = h_A[i] - h_B[i];
    }
    clock_t cpu_end = clock();
    double cpu_time = (double)(cpu_end - cpu_start) / CLOCKS_PER_SEC * 1000.0;

    // Выделение памяти на устройстве
    float *d_A, *d_B, *d_C;
    cudaMalloc(&d_A, bytes);
    cudaMalloc(&d_B, bytes);
    cudaMalloc(&d_C, bytes);

    // Копирование данных на устройство
    cudaMemcpy(d_A, h_A, bytes, cudaMemcpyHostToDevice);
    cudaMemcpy(d_B, h_B, bytes, cudaMemcpyHostToDevice);

    // Настройка сетки и блоков
    int threadsPerBlock = 256;
    int blocksPerGrid = (N + threadsPerBlock - 1) / threadsPerBlock;

    // Замер времени GPU (копирование + ядро + копирование обратно)
    cudaEvent_t start, stop;
    cudaEventCreate(&start);
    cudaEventCreate(&stop);
    cudaEventRecord(start);

    // Запуск ядра
    vectorSubtract<<<blocksPerGrid, threadsPerBlock>>>(d_A, d_B, d_C, N);
    cudaDeviceSynchronize();

    // Копирование результата обратно
    cudaMemcpy(h_C_gpu, d_C, bytes, cudaMemcpyDeviceToHost);

    cudaEventRecord(stop);
    cudaEventSynchronize(stop);
    float gpu_time = 0;
    cudaEventElapsedTime(&gpu_time, start, stop);

    // Проверка корректности
    bool isCorrect = verifyResult(h_C_cpu, h_C_gpu, N);
    printf("\n=== Results ===\n");
    printf("Vector size: %d\n", N);
    printf("CPU time: %.2f ms\n", cpu_time);
    printf("GPU time (copy + kernel + copy back): %.2f ms\n", gpu_time);
    printf("Results match: %s\n", isCorrect ? "TRUE" : "FALSE");

    // Освобождение памяти
    cudaFree(d_A);
    cudaFree(d_B);
    cudaFree(d_C);
    free(h_A);
    free(h_B);
    free(h_C_cpu);
    free(h_C_gpu);

    return 0;
}

Writing vector_subtract.cu


In [ ]:
!nvcc -arch=compute_75 -code=sm_75 \
      -gencode arch=compute_75,code=sm_75 \
      vector_subtract.cu -o vector_subtract

In [ ]:
!./vector_subtract

Enter vector size (e.g., 1000000): 1000000

=== Results ===
Vector size: 1000000
CPU time: 4.67 ms
GPU time (copy + kernel + copy back): 3.12 ms
Results match: TRUE


In [ ]:
# @title
%%writefile char_count.cu

#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <time.h>

#define ALPHABET_SIZE 256

// CUDA kernel для подсчёта символов с использованием атомарных операций
__global__ void countChars(const unsigned char *text, int *counts, int textLength) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    if (idx < textLength) {
        unsigned char ch = text[idx];
        atomicAdd(&counts[ch], 1);
    }
}

// CPU реализация для проверки
void countCharsCPU(const unsigned char *text, int *counts, int textLength) {
    for (int i = 0; i < textLength; i++) {
        counts[text[i]]++;
    }
}

// Генерация случайного текста
unsigned char* generateRandomText(int length) {
    unsigned char *text = (unsigned char*)malloc(length);
    for (int i = 0; i < length; i++) {
        text[i] = rand() % ALPHABET_SIZE;
    }
    return text;
}

// Проверка результатов
bool verifyResults(const int *cpu_counts, const int *gpu_counts) {
    for (int i = 0; i < ALPHABET_SIZE; i++) {
        if (cpu_counts[i] != gpu_counts[i]) {
            printf("Mismatch at char %d: CPU = %d, GPU = %d\n", i, cpu_counts[i], gpu_counts[i]);
            return false;
        }
    }
    return true;
}

int main() {
    int textLength;
    printf("Enter text length (e.g., 4000000): ");
    scanf("%d", &textLength);

    // Генерация текста
    srand(time(NULL));
    unsigned char *h_text = generateRandomText(textLength);
    size_t textBytes = textLength * sizeof(unsigned char);
    size_t countsBytes = ALPHABET_SIZE * sizeof(int);

    // CPU подсчёт
    int *h_cpu_counts = (int*)calloc(ALPHABET_SIZE, sizeof(int));
    clock_t cpu_start = clock();
    countCharsCPU(h_text, h_cpu_counts, textLength);
    clock_t cpu_end = clock();
    double cpu_time = (double)(cpu_end - cpu_start) / CLOCKS_PER_SEC * 1000.0;

    // Выделение памяти на GPU
    unsigned char *d_text;
    int *d_counts;
    cudaMalloc(&d_text, textBytes);
    cudaMalloc(&d_counts, countsBytes);
    cudaMemset(d_counts, 0, countsBytes);

    // Копирование текста на GPU
    cudaMemcpy(d_text, h_text, textBytes, cudaMemcpyHostToDevice);

    // Настройка сетки и блоков
    int threadsPerBlock = 256;
    int blocksPerGrid = (textLength + threadsPerBlock - 1) / threadsPerBlock;

    // Замер времени GPU
    cudaEvent_t start, stop;
    cudaEventCreate(&start);
    cudaEventCreate(&stop);
    cudaEventRecord(start);

    // Запуск ядра
    countChars<<<blocksPerGrid, threadsPerBlock>>>(d_text, d_counts, textLength);
    cudaDeviceSynchronize();

    // Копирование результатов обратно
    int *h_gpu_counts = (int*)malloc(countsBytes);
    cudaMemcpy(h_gpu_counts, d_counts, countsBytes, cudaMemcpyDeviceToHost);

    cudaEventRecord(stop);
    cudaEventSynchronize(stop);
    float gpu_time = 0;
    cudaEventElapsedTime(&gpu_time, start, stop);

    // Проверка корректности
    bool isCorrect = verifyResults(h_cpu_counts, h_gpu_counts);

    // Вывод результатов (только первые 20 символов для наглядности)
    printf("\n=== Results ===\n");
    printf("Text length: %d characters\n", textLength);
    printf("CPU time: %.2f ms\n", cpu_time);
    printf("GPU time (copy + kernel + copy back): %.2f ms\n", gpu_time);
    printf("Results match: %s\n", isCorrect ? "TRUE" : "FALSE");

    printf("\nSample character counts (first 20):\n");
    for (int i = 0; i < 20; i++) {
        printf("Char %3d: CPU = %5d, GPU = %5d\n", i, h_cpu_counts[i], h_gpu_counts[i]);
    }

    // Освобождение памяти
    cudaFree(d_text);
    cudaFree(d_counts);
    free(h_text);
    free(h_cpu_counts);
    free(h_gpu_counts);

    return 0;
}

Writing char_count.cu


In [ ]:
!nvcc -arch=compute_75 -code=sm_75 \
      -gencode arch=compute_75,code=sm_75 \
      char_count.cu -o char_count

In [ ]:
!./char_count

Enter text length (e.g., 4000000): 4000000

=== Results ===
Text length: 4000000 characters
CPU time: 8.41 ms
GPU time (copy + kernel + copy back): 2.34 ms
Results match: TRUE

Sample character counts (first 20):
Char   0: CPU = 15596, GPU = 15596
Char   1: CPU = 15466, GPU = 15466
Char   2: CPU = 15529, GPU = 15529
Char   3: CPU = 15610, GPU = 15610
Char   4: CPU = 15507, GPU = 15507
Char   5: CPU = 15509, GPU = 15509
Char   6: CPU = 15428, GPU = 15428
Char   7: CPU = 15877, GPU = 15877
Char   8: CPU = 15460, GPU = 15460
Char   9: CPU = 15349, GPU = 15349
Char  10: CPU = 15536, GPU = 15536
Char  11: CPU = 15594, GPU = 15594
Char  12: CPU = 15754, GPU = 15754
Char  13: CPU = 15563, GPU = 15563
Char  14: CPU = 15538, GPU = 15538
Char  15: CPU = 15728, GPU = 15728
Char  16: CPU = 15763, GPU = 15763
Char  17: CPU = 15712, GPU = 15712
Char  18: CPU = 15969, GPU = 15969
Char  19: CPU = 15652, GPU = 15652


In [5]:
%%writefile matrix_mul.cu

#include <iostream>
#include <vector>
#include <random>
#include <chrono>
#include <cmath>
#include <cuda_runtime.h>

#define CUDA_CHECK(call) \
    do { \
        cudaError_t status = (call); \
        if (status != cudaSuccess) { \
            std::cerr << "CUDA Error: " << cudaGetErrorString(status) << "\n"; \
            std::exit(EXIT_FAILURE); \
        } \
    } while (0)

constexpr int BLOCK_SZ = 16;

void multiplyOnCPU(const float* mtxA, const float* mtxB, float* result, int rows, int innerDim)
{
    for (int i = 0; i < rows; ++i) {
        for (int j = 0; j < rows; ++j) {
            float accumulator = 0.0f;
            for (int k = 0; k < innerDim; ++k) {
                accumulator += mtxA[i * innerDim + k] * mtxB[k * rows + j];
            }
            result[i * rows + j] = accumulator;
        }
    }
}


bool verifyResults(const std::vector<float>& cpuRes, const std::vector<float>& gpuRes)
{
    constexpr float tolerance = 1e-2f;
    for (size_t idx = 0; idx < cpuRes.size(); ++idx) {
        if (std::fabs(cpuRes[idx] - gpuRes[idx]) > tolerance) {
            return false;
        }
    }
    return true;
}

__global__ void multiplyOnGPU(const float* __restrict__ A,
    const float* __restrict__ B,
    float* C,
    int n, int m)
{
    __shared__ float tileA[BLOCK_SZ][BLOCK_SZ];
    __shared__ float tileB[BLOCK_SZ][BLOCK_SZ];

    int rowIdx = blockIdx.y * BLOCK_SZ + threadIdx.y;
    int colIdx = blockIdx.x * BLOCK_SZ + threadIdx.x;

    float partialSum = 0.0f;
    int tileCount = (m + BLOCK_SZ - 1) / BLOCK_SZ;

    for (int tile = 0; tile < tileCount; ++tile)
    {
        int aRow = rowIdx;
        int aCol = tile * BLOCK_SZ + threadIdx.x;
        tileA[threadIdx.y][threadIdx.x] = (aRow < n && aCol < m) ? A[aRow * m + aCol] : 0.0f;

        int bRow = tile * BLOCK_SZ + threadIdx.y;
        int bCol = colIdx;
        tileB[threadIdx.y][threadIdx.x] = (bRow < m && bCol < n) ? B[bRow * n + bCol] : 0.0f;

        __syncthreads();

#pragma unroll
        for (int k = 0; k < BLOCK_SZ; ++k) {
            partialSum += tileA[threadIdx.y][k] * tileB[k][threadIdx.x];
        }

        __syncthreads();
    }

    if (rowIdx < n && colIdx < n) {
        C[rowIdx * n + colIdx] = partialSum;
    }
}

int main()
{
    int dimN, dimM;
    std::cout << "Enter matrix dimensions (n m): ";
    std::cin >> dimN >> dimM;

    const size_t countA = static_cast<size_t>(dimN) * dimM;
    const size_t countB = static_cast<size_t>(dimM) * dimN;
    const size_t countC = static_cast<size_t>(dimN) * dimN;

    std::cout << "Matrix A size: " << countA << " elements\n";
    std::cout << "Matrix B size: " << countB << " elements\n";

    std::vector<float> h_A(countA), h_B(countB);
    std::vector<float> h_C_cpu(countC), h_C_gpu(countC);

    std::mt19937 rng(777);
    std::uniform_real_distribution<float> uniform(-5.0f, 5.0f);
    for (auto& val : h_A) val = uniform(rng);
    for (auto& val : h_B) val = uniform(rng);

    auto cpuStart = std::chrono::high_resolution_clock::now();
    multiplyOnCPU(h_A.data(), h_B.data(), h_C_cpu.data(), dimN, dimM);
    auto cpuEnd = std::chrono::high_resolution_clock::now();
    double cpuElapsed = std::chrono::duration<double, std::milli>(cpuEnd - cpuStart).count();

    float* d_A = nullptr, * d_B = nullptr, * d_C = nullptr;
    CUDA_CHECK(cudaMalloc(&d_A, countA * sizeof(float)));
    CUDA_CHECK(cudaMalloc(&d_B, countB * sizeof(float)));
    CUDA_CHECK(cudaMalloc(&d_C, countC * sizeof(float)));

    cudaEvent_t evtStart, evtStop;
    cudaEventCreate(&evtStart);
    cudaEventCreate(&evtStop);

    cudaEventRecord(evtStart);

    CUDA_CHECK(cudaMemcpy(d_A, h_A.data(), countA * sizeof(float), cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(d_B, h_B.data(), countB * sizeof(float), cudaMemcpyHostToDevice));

    dim3 threadsPerBlock(BLOCK_SZ, BLOCK_SZ);
    dim3 blocksPerGrid((dimN + BLOCK_SZ - 1) / BLOCK_SZ,
        (dimN + BLOCK_SZ - 1) / BLOCK_SZ);

    multiplyOnGPU << <blocksPerGrid, threadsPerBlock >> > (d_A, d_B, d_C, dimN, dimM);
    CUDA_CHECK(cudaDeviceSynchronize());

    CUDA_CHECK(cudaMemcpy(h_C_gpu.data(), d_C, countC * sizeof(float), cudaMemcpyDeviceToHost));

    cudaEventRecord(evtStop);
    cudaEventSynchronize(evtStop);

    float gpuElapsed = 0.0f;
    cudaEventElapsedTime(&gpuElapsed, evtStart, evtStop);

    bool isCorrect = verifyResults(h_C_cpu, h_C_gpu);

    std::cout << "\n=== Report ===\n";
    std::cout << "CPU execution time: " << cpuElapsed << " ms\n";
    std::cout << "GPU time: " << gpuElapsed << " ms\n";
    std::cout << "VALID: " << (isCorrect ? "TRUE" : "FALSE") << "\n";

    cudaFree(d_A);
    cudaFree(d_B);
    cudaFree(d_C);
    cudaEventDestroy(evtStart);
    cudaEventDestroy(evtStop);

    return 0;
}

Overwriting matrix_mul.cu


In [6]:
!nvcc -arch=compute_75 -code=sm_75 \
      -gencode arch=compute_75,code=sm_75 \
      matrix_mul.cu -o matrix_mul

In [7]:
!./matrix_mul

Enter matrix dimensions (n m): 2500 2500
Matrix A size: 6250000 elements
Matrix B size: 6250000 elements

=== Report ===
CPU execution time: 170647 ms
GPU time: 76.2931 ms
VALID: TRUE
